# Dispersion figures from `pinn_data.mat`
Self-contained notebook that generates the same four figure types in one combined panel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
mat = loadmat('../Result/pinn_data.mat')
print('Available keys:', [k for k in mat.keys() if not k.startswith('__')])

preferred = ['x', 'disp', 'u', 'response', 'X', 'U']
X = None
selected_key = None
for key in preferred:
    if key in mat:
        arr = np.array(mat[key])
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            X, selected_key = arr, key
            break

if X is None:
    candidates = []
    for k, v in mat.items():
        if k.startswith('__'):
            continue
        arr = np.array(v)
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            candidates.append((k, arr))
    if not candidates:
        raise ValueError('No numeric 2D+ array found in pinn_data.mat.')
    selected_key, X = max(candidates, key=lambda kv: kv[1].shape[0] * kv[1].shape[1])

if X.ndim > 2:
    X = np.reshape(X, (X.shape[0], -1))
if X.shape[0] < X.shape[1]:
    X = X.T

X = X - X.mean(axis=0, keepdims=True)
nt, nx = X.shape
dt = float(np.squeeze(mat['dt'])) if 'dt' in mat else 1.0
dx = float(np.squeeze(mat['dx'])) if 'dx' in mat else 1.0
time = np.arange(nt) * dt
space = np.arange(nx) * dx
print('Using key:', selected_key, '| shape:', X.shape, '| dt:', dt, '| dx:', dx)

In [ ]:
S = np.fft.fftshift(np.fft.fft2(X))
P = np.abs(S)**2
f = np.fft.fftshift(np.fft.fftfreq(nt, d=dt))
k = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(nx, d=dx))

# Use one-sided (non-negative) frequency and wavenumber for cleaner physical view
mask_f = f >= 0
mask_k = k >= 0
f_pos = f[mask_f]
k_pos = k[mask_k]
P_pos = P[np.ix_(mask_f, mask_k)]
P_db_pos = 10*np.log10(P_pos/(P_pos.max()+1e-12) + 1e-12)

k_ridge = k_pos[np.argmax(P_pos, axis=1)]

## Combined 2x2 figure panel (same outputs)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# (1) time-space field
ax = axes[0, 0]
im1 = ax.pcolormesh(space, time, X, shading='auto', cmap='RdBu_r')
ax.set_title('01_time_space_field')
ax.set_xlabel('Space index / coordinate')
ax.set_ylabel('Time')
fig.colorbar(im1, ax=ax, label='Displacement')

# (2) time traces
ax = axes[0, 1]
for j in np.linspace(0, nx-1, min(4, nx), dtype=int):
    ax.plot(time, X[:, j], label=f'space #{j}')
ax.set_title('02_time_traces')
ax.set_xlabel('Time')
ax.set_ylabel('Displacement')
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

# (3) one-sided f-k map (k >= 0, f >= 0)
ax = axes[1, 0]
im2 = ax.pcolormesh(k_pos, f_pos, P_db_pos, shading='auto', cmap='magma')
ax.set_title('03_fk_map (one-sided)')
ax.set_xlabel('Wavenumber k [rad/unit]')
ax.set_ylabel('Frequency f [1/unit]')
fig.colorbar(im2, ax=ax, label='Power (dB, normalized)')

# (4) dispersion ridge from one-sided map
ax = axes[1, 1]
ax.plot(k_ridge, f_pos, '.', ms=3)
ax.set_title('04_dispersion_ridge')
ax.set_xlabel('Dominant wavenumber k')
ax.set_ylabel('Frequency f')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Note: negative wavenumbers are mathematically valid in FFT (opposite propagation direction). This notebook now shows one-sided k>=0 only.